In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
from zipfile import ZipFile

zip_file = "appliances+energy+prediction.zip"

with ZipFile(zip_file, 'r') as zip:
    zip.extractall()

print("CSV extracted successfully")

CSV extracted successfully


In [3]:
import pandas as pd

df = pd.read_csv("energydata_complete.csv")

df.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  object 
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7         19735 non-null  float64
 17  T8           19735 non-null  float64
 18  RH_8         19735 non-null  float64
 19  T9  

In [5]:
df.isnull().sum()

date           0
Appliances     0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH_8           0
T9             0
RH_9           0
T_out          0
Press_mm_hg    0
RH_out         0
Windspeed      0
Visibility     0
Tdewpoint      0
rv1            0
rv2            0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df["date"] = pd.to_datetime(df["date"])

df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["hour"] = df["date"].dt.hour
df["minute"] = df["date"].dt.minute
df["dayofweek"] = df["date"].dt.dayofweek

In [8]:
df.drop(columns=["date"], inplace=True)

In [12]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

def cap_outliers(df, cols):
    df = df.copy()
    
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        df[col] = df[col].clip(lower, upper)
    
    return df

df= cap_outliers(df, num_cols)

In [13]:
y=df["Appliances"]

In [14]:
x=df.drop(["Appliances"],axis=1)

In [15]:
# categorical columns
cat_cols = x.select_dtypes(include=["object"]).columns

# numerical columns
num_cols = x.select_dtypes(include=["int64", "float64"]).columns

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
ct = ColumnTransformer(
    transformers=[
        ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
        ("scaler", StandardScaler(), num_cols)
    ],
    remainder="passthrough"
)

In [17]:
x_transformed = ct.fit_transform(x)
x_transformed = pd.DataFrame(x_transformed)

print(x_transformed.head())

    0         1         2         3         4         5         6         7   \
0  0.0 -1.139072  1.863478 -0.528718  1.092582 -1.245155  1.686863 -0.912635   
1  0.0 -1.139072  1.634348 -0.528718  1.075633 -1.245155  1.705307 -0.912635   
2  0.0 -1.139072  1.534580 -0.528718  1.051570 -1.245155  1.749367 -0.948663   
3  0.0 -1.139072  1.475395 -0.528718  1.042363 -1.245155  1.769859 -0.966677   
4  0.0 -1.139072  1.543034 -0.528718  1.027297 -1.245155  1.769859 -0.966677   

         8         9   ...        23        24        25        26    27   28  \
0  1.506438 -1.319168  ...  1.799947  0.367016 -0.807974 -0.807974  11.0  1.0   
1  1.604528 -1.319168  ...  1.799947  0.343175 -0.440240 -0.440240  11.0  1.0   
2  1.580918 -1.319168  ...  1.687977  0.319333  0.252109  0.252109  11.0  1.0   
3  1.542526 -1.319168  ...  1.320076  0.295491  1.408801  1.408801  11.0  1.0   
4  1.497991 -1.301016  ...  0.952175  0.271649 -1.028122 -1.028122  11.0  1.0   

       29    30    31   32  
0  

In [18]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [20]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [22]:
y_pred=model.predict(x_test)

In [23]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

print("R2 Score:", r2)

R2 Score: 0.25677901218401955


In [25]:
# Ridge Regression
from sklearn.linear_model import Ridge

# Lasso Regression
from sklearn.linear_model import Lasso

In [26]:
ls=Lasso()

In [28]:
ls.fit(x_train,y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [29]:
y_pred=ls.predict(x_test)

In [30]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

print("R2 Score:", r2)

R2 Score: 0.23909709571569637


In [31]:
ridge = Ridge()

In [33]:
ridge.fit(x_train,y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [35]:
y_pred_ridge = ridge.predict(x_test)

In [36]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

print("R2 Score:", r2)

R2 Score: 0.23909709571569637
